In [1]:
import pandas

In [2]:
!pip install pyspark

     ---------------------------------------- 0.0/455.4 MB ? eta -:--:--
     ---------------------------------------- 0.0/455.4 MB ? eta -:--:--
     ---------------------------------------- 0.0/455.4 MB ? eta -:--:--
     ---------------------------------------- 0.3/455.4 MB ? eta -:--:--
     ---------------------------------------- 0.8/455.4 MB 1.4 MB/s eta 0:05:26
     ---------------------------------------- 2.1/455.4 MB 2.9 MB/s eta 0:02:35
      --------------------------------------- 6.0/455.4 MB 6.7 MB/s eta 0:01:07
      --------------------------------------- 8.9/455.4 MB 8.8 MB/s eta 0:00:51
      --------------------------------------- 8.9/455.4 MB 8.8 MB/s eta 0:00:51
      --------------------------------------- 8.9/455.4 MB 8.8 MB/s eta 0:00:51
      --------------------------------------- 8.9/455.4 MB 8.8 MB/s eta 0:00:51
      --------------------------------------- 8.9/455.4 MB 8.8 MB/s eta 0:00:51
      --------------------------------------- 9.2/455.4 MB 4.5 MB/s 

In [2]:
import pyspark as ps
import pandas as pd
import numpy as np

In [48]:
df=pd.read_csv("owid-covid-latest.csv")

In [49]:
# according to requirement the columns i need are "location","date","total_cases","total_deaths", "new_cases","new_deaths"
# so we find the null values in these columns
df_1=df[["location","last_updated_date","total_cases","total_deaths", "new_cases","new_deaths"]]
df_1.isnull().sum()

location             0
last_updated_date    0
total_cases          1
total_deaths         1
new_cases            5
new_deaths           4
dtype: int64

In [50]:
# As the columns having missing values are numeric so we fill them with median of that column
df["total_cases"].fillna(df["total_cases"].median(), inplace=True)
df["total_deaths"].fillna(df["total_deaths"].median(), inplace=True)
df["new_cases"].fillna(df["new_cases"].median(), inplace=True)
df["new_deaths"].fillna(df["new_deaths"].median(), inplace=True)

C:\Users\MS\AppData\Local\Temp\ipykernel_13688\3813316432.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["total_cases"].fillna(df["total_cases"].median(), inplace=True)
C:\Users\MS\AppData\Local\Temp\ipykernel_13688\3813316432.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a co

In [51]:
df[["location","last_updated_date","total_cases","total_deaths", "new_cases","new_deaths"]].isnull().sum()

location             0
last_updated_date    0
total_cases          0
total_deaths         0
new_cases            0
new_deaths           0
dtype: int64

In [52]:
# save the updated csv
df.to_csv("owid-covid-latest-updated.csv", index=False)

0

# Introduction

COVID-19 affected countries worldwide. This project analyzes global COVID-19 statistics using PySpark for efficient big data processing.

In [4]:
# Import libraries
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, max

# Create Spark session
spark = SparkSession.builder \
    .appName("COVID19 Analysis") \
    .getOrCreate()

In [53]:

# Load dataset
df = spark.read.csv("owid-covid-latest-updated.csv",
                    header=True,
                    inferSchema=True)


In [54]:
# Show dataset structure
df.printSchema()

root
 |-- iso_code: string (nullable = true)
 |-- continent: string (nullable = true)
 |-- location: string (nullable = true)
 |-- last_updated_date: date (nullable = true)
 |-- total_cases: double (nullable = true)
 |-- new_cases: double (nullable = true)
 |-- new_cases_smoothed: double (nullable = true)
 |-- total_deaths: double (nullable = true)
 |-- new_deaths: double (nullable = true)
 |-- new_deaths_smoothed: double (nullable = true)
 |-- total_cases_per_million: double (nullable = true)
 |-- new_cases_per_million: double (nullable = true)
 |-- new_cases_smoothed_per_million: double (nullable = true)
 |-- total_deaths_per_million: double (nullable = true)
 |-- new_deaths_per_million: double (nullable = true)
 |-- new_deaths_smoothed_per_million: double (nullable = true)
 |-- reproduction_rate: string (nullable = true)
 |-- icu_patients: double (nullable = true)
 |-- icu_patients_per_million: double (nullable = true)
 |-- hosp_patients: double (nullable = true)
 |-- hosp_patients_

In [55]:
# Show first 5 rows
df.show(5)

+--------+---------+--------------+-----------------+-----------+---------+------------------+------------+----------+-------------------+-----------------------+---------------------+------------------------------+------------------------+----------------------+-------------------------------+-----------------+------------+------------------------+-------------+-------------------------+---------------------+---------------------------------+----------------------+----------------------------------+-----------+---------+------------------------+----------------------+------------------+-------------------------------+-------------+--------------+-----------+------------------+-----------------+-----------------------+--------------+----------------+-------------------------+------------------------------+-----------------------------+-----------------------------------+--------------------------+-------------------------------------+------------------------------+---------------------

In [56]:
# shows the countries name inm dataset
df.select("location").distinct().show()

+-----------+
|   location|
+-----------+
|       Chad|
|   Anguilla|
|   Paraguay|
|     Russia|
|      World|
|      Yemen|
|    Senegal|
|     Sweden|
|    Tokelau|
|   Kiribati|
|     Guyana|
|    Eritrea|
|     Jersey|
|Philippines|
|   Djibouti|
|      Tonga|
|   Malaysia|
|  Singapore|
|       Fiji|
|     Turkey|
+-----------+
only showing top 20 rows


In [70]:
# show the 5 rows where location = sweden
sweden_df = df.filter(col("location") == "Kiribati")
sweden_df.select(
    "last_updated_date",
    "total_cases",
    "total_deaths",
    "new_cases",
    "new_deaths"
).show()

+-----------------+-----------+------------+---------+----------+
|last_updated_date|total_cases|total_deaths|new_cases|new_deaths|
+-----------------+-----------+------------+---------+----------+
|       2024-08-04|     5085.0|        24.0|      0.0|       0.0|
+-----------------+-----------+------------+---------+----------+



In [71]:
# Find total cases and deaths
total_stats = df.groupBy("location").agg(
    max("total_cases").alias("Total_Cases"),
    max("total_deaths").alias("Total_Deaths")
)

total_stats.show()

+-----------+------------+------------+
|   location| Total_Cases|Total_Deaths|
+-----------+------------+------------+
|       Chad|      7702.0|       194.0|
|   Anguilla|      3904.0|        12.0|
|   Paraguay|    735759.0|     19880.0|
|     Russia| 2.4268728E7|    403188.0|
|      World|7.75866783E8|   7057132.0|
|      Yemen|     11945.0|      2159.0|
|    Senegal|     89485.0|      1971.0|
|     Sweden|   2755181.0|     27399.0|
|    Tokelau|        80.0|         0.0|
|   Kiribati|      5085.0|        24.0|
|     Guyana|     74443.0|      1302.0|
|    Eritrea|     10189.0|       103.0|
|     Jersey|     66391.0|       161.0|
|Philippines|   4140383.0|     66864.0|
|   Djibouti|     15690.0|       189.0|
|      Tonga|     16976.0|        12.0|
|   Malaysia|   5309410.0|     37351.0|
|  Singapore|   3006155.0|      2024.0|
|       Fiji|     69047.0|       885.0|
|     Turkey| 1.7004718E7|    101419.0|
+-----------+------------+------------+
only showing top 20 rows


In [75]:
# Sort by highest cases
highest_cases = total_stats.orderBy(
    col("total_cases").desc()
)

highest_cases.show(10)

+--------------------+------------+------------+
|            location| Total_Cases|Total_Deaths|
+--------------------+------------+------------+
|               World|7.75866783E8|   7057132.0|
|High-income count...|4.29044049E8|   2997359.0|
|                Asia|3.01499099E8|   1637249.0|
|              Europe|2.52916868E8|   2102483.0|
|Upper-middle-inco...|2.51753518E8|   2824452.0|
| European Union (27)|1.85822587E8|   1262988.0|
|       North America|1.24492666E8|   1671178.0|
|       United States|1.03436829E8|   1193165.0|
|               China| 9.9373219E7|    122304.0|
|Lower-middle-inco...|   9.19544E7|   1188026.0|
+--------------------+------------+------------+
only showing top 10 rows


# Insights (Bullet Points)
1. The high income countries reported the highest number of COVID-19 cases. 
2. India and Brazil were among the most affected countries.
3. Some countries had high case counts but relatively lower death rates.
4. Daily new cases fluctuated significantly during pandemic waves.
5. PySpark efficiently processed large-scale COVID datasets.

# Conclusion

This project demonstrated how PySpark can be used for large-scale COVID-19 data analysis. Using DataFrames and aggregation functions, we extracted useful insights from real-world pandemic data.